# Initial Setup - Colab Enterprise

**Purpose:** Complete environment setup for ML badminton coaching project

**Duration:** ~10-20 minutes

**This notebook sets up:**
1. ✅ Repository cloning
2. ✅ Python 3.10 virtual environment
3. ✅ Google Cloud Storage authentication
4. ✅ Directory structure
5. ✅ Video access verification
6. ✅ Dependencies installation

**Prerequisites:**
- Google Cloud account
- GCS bucket created (e.g., `gs://iti123storage`)
- Service account JSON key OR Colab authentication
- Videos uploaded to `gs://YOUR_BUCKET/videos/clips/`

**After this notebook:**
- Environment ready for Phase 2 (pose extraction)
- All dependencies installed
- GCS access verified

---

## Step 1: Check Colab Environment

Verify we're running in Colab and check available resources.

In [ ]:
# Check Python version and environment
import sys
import os

print(f"Python version: {sys.version}")
print(f"Current directory: {os.getcwd()}")
print(f"\nNote: Colab defaults to Python 3.12")
print(f"We'll create a Python 3.10 virtual environment for compatibility")

In [ ]:
# Check GPU availability
!nvidia-smi || echo "No GPU available (CPU-only runtime)"

In [ ]:
# Check available disk space
!df -h /content

**✅ Checkpoint 1:** Environment checked

---

## Step 2: Clone Repository

Clone the project repository to `/content/iti123_v2`

In [ ]:
# Change to /content directory
%cd /content

# Remove existing directory if present
!rm -rf iti123_v2

# Clone repository
# Replace YOUR_USERNAME with your GitHub username
GITHUB_USERNAME = "YOUR_USERNAME"  # ⚠️ CHANGE THIS!

!git clone https://github.com/{GITHUB_USERNAME}/iti123_v2.git

# Verify clone
if os.path.exists('/content/iti123_v2'):
    print("\n✓ Repository cloned successfully")
    %cd /content/iti123_v2
    !git branch
else:
    print("\n❌ Clone failed - check GitHub username and repository access")

In [ ]:
# Checkout the correct branch
!git checkout milestone/v1.1-coach-informed-ml
!git pull origin milestone/v1.1-coach-informed-ml

print("\n✓ On correct branch")

**✅ Checkpoint 2:** Repository cloned

---

## Step 3: Setup Python 3.10 Environment

Create a Python 3.10 virtual environment for TensorFlow 2.15 compatibility.

In [ ]:
# Run setup script
print("Creating Python 3.10 virtual environment...")
print("This will take 3-5 minutes\n")

!bash scripts/colab_setup.sh

In [ ]:
# Verify virtual environment
!colab_venv/bin/python --version
!colab_venv/bin/pip --version

print("\n✓ Python 3.10 environment ready")

In [ ]:
# Verify key packages
print("Verifying installed packages...\n")

!colab_venv/bin/python -c "import tensorflow as tf; print(f'✓ TensorFlow: {tf.__version__}')"
!colab_venv/bin/python -c "import mediapipe as mp; print('✓ MediaPipe: installed')"
!colab_venv/bin/python -c "import sklearn; print('✓ Scikit-learn: installed')"
!colab_venv/bin/python -c "import pandas; print('✓ Pandas: installed')"
!colab_venv/bin/python -c "import numpy; print('✓ NumPy: installed')"

print("\n✓ All required packages installed")

**✅ Checkpoint 3:** Python environment ready

**Important:** For the rest of this notebook and all subsequent work, use `colab_venv/bin/python` instead of `python`

---

## Step 4: Authenticate with Google Cloud Storage

**Choose ONE of the following methods:**

### Method A: Service Account Key (Recommended for Production)

Upload your service account JSON key file.

In [ ]:
# Upload service account key
from google.colab import files

print("Please upload your service account JSON key file")
print("(This will open a file picker)\n")

uploaded = files.upload()

if uploaded:
    # Get the filename
    key_filename = list(uploaded.keys())[0]
    key_path = f'/content/{key_filename}'
    
    # Set environment variable
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = key_path
    
    print(f"\n✓ Service account key uploaded: {key_filename}")
    print(f"✓ GOOGLE_APPLICATION_CREDENTIALS set to: {key_path}")
    
    # Save for later use
    with open('/content/gcs_key_path.txt', 'w') as f:
        f.write(key_path)
else:
    print("\n⚠️  No file uploaded - trying Method B instead")

### Method B: Colab Default Authentication (Alternative)

Use Colab's built-in authentication. **Only run this if Method A didn't work.**

In [ ]:
# Uncomment and run if you prefer this method

# from google.colab import auth
# auth.authenticate_user()
# print("✓ Authenticated with Colab default credentials")

**✅ Checkpoint 4:** GCS authentication configured

---

## Step 5: Configure GCS Bucket

Set your GCS bucket name and verify access.

In [ ]:
# Set your GCS bucket name
GCS_BUCKET = "iti123storage"  # ⚠️ CHANGE THIS to your bucket name!

print(f"GCS Bucket: gs://{GCS_BUCKET}")

# Save for later use
with open('/content/gcs_bucket_name.txt', 'w') as f:
    f.write(GCS_BUCKET)

print(f"\n✓ Bucket name saved")

In [ ]:
# Test GCS access
print(f"Testing access to gs://{GCS_BUCKET}/\n")

result = !gsutil ls gs://{GCS_BUCKET}/

if result:
    print("\n✓ GCS access verified")
    print("\nBucket contents:")
    for item in result:
        print(f"  {item}")
else:
    print("\n❌ Cannot access bucket")
    print("\nTroubleshooting:")
    print("1. Check bucket name is correct")
    print("2. Verify service account has Storage Admin role")
    print("3. Check GOOGLE_APPLICATION_CREDENTIALS is set")
    print(f"   Current: {os.getenv('GOOGLE_APPLICATION_CREDENTIALS', 'Not set')}")

**✅ Checkpoint 5:** GCS bucket configured and accessible

---

## Step 6: Verify Video Files

Check that videos are available in GCS.

In [ ]:
# Check for video clips
print("Checking for video files in GCS...\n")

# Count clear videos
clear_result = !gsutil ls gs://{GCS_BUCKET}/videos/clips/clear/*.mp4 2>/dev/null | wc -l
clear_count = int(clear_result[0]) if clear_result else 0

# Count smash videos
smash_result = !gsutil ls gs://{GCS_BUCKET}/videos/clips/smash/*.mp4 2>/dev/null | wc -l
smash_count = int(smash_result[0]) if smash_result else 0

total = clear_count + smash_count

print(f"Clear videos: {clear_count}")
print(f"Smash videos: {smash_count}")
print(f"Total: {total}")

if total == 0:
    print("\n⚠️  No videos found!")
    print("\nExpected structure:")
    print(f"  gs://{GCS_BUCKET}/videos/clips/clear/*.mp4")
    print(f"  gs://{GCS_BUCKET}/videos/clips/smash/*.mp4")
    print("\nPlease upload videos before proceeding.")
elif total < 100:
    print(f"\n⚠️  Only {total} videos found")
    print("This is enough for testing, but you may want more for production.")
    print(f"✓ Videos accessible")
else:
    print(f"\n✓ Found {total} videos - ready for pose extraction!")

In [ ]:
# Show sample video paths (first 5 of each type)
if total > 0:
    print("Sample clear videos:")
    !gsutil ls gs://{GCS_BUCKET}/videos/clips/clear/*.mp4 | head -5
    
    print("\nSample smash videos:")
    !gsutil ls gs://{GCS_BUCKET}/videos/clips/smash/*.mp4 | head -5

**✅ Checkpoint 6:** Videos verified in GCS

---

## Step 7: Create Directory Structure

Set up local directories for data processing.

In [ ]:
# Create directory structure
directories = [
    'data/videos/clips/clear',
    'data/videos/clips/smash',
    'data/processed/poses',
    'data/processed/features_v3',
    'models/v3',
    'outputs/reports'
]

for directory in directories:
    !mkdir -p {directory}

print("✓ Directory structure created")

# Show structure
print("\nDirectory tree:")
!tree -L 3 data/ models/ outputs/ 2>/dev/null || find data/ models/ outputs/ -type d | head -20

**✅ Checkpoint 7:** Directories created

---

## Step 8: Environment Summary

Display complete setup status.

In [ ]:
# Generate setup summary
import json
from datetime import datetime

setup_info = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'working_directory': os.getcwd(),
    'python_version': sys.version.split()[0],
    'venv_python': !colab_venv/bin/python --version,
    'gcs_bucket': GCS_BUCKET,
    'gcs_key_path': os.getenv('GOOGLE_APPLICATION_CREDENTIALS', 'Using Colab auth'),
    'video_count': {
        'clear': clear_count,
        'smash': smash_count,
        'total': total
    },
    'gpu_available': os.system('nvidia-smi > /dev/null 2>&1') == 0
}

print(f"{'='*60}")
print("SETUP SUMMARY")
print(f"{'='*60}")
print(f"Timestamp: {setup_info['timestamp']}")
print(f"\nEnvironment:")
print(f"  Working directory: {setup_info['working_directory']}")
print(f"  System Python: {setup_info['python_version']}")
print(f"  Venv Python: {setup_info['venv_python'][0].split()[-1]}")
print(f"  GPU available: {'Yes' if setup_info['gpu_available'] else 'No (CPU only)'}")

print(f"\nGoogle Cloud Storage:")
print(f"  Bucket: gs://{setup_info['gcs_bucket']}")
print(f"  Auth: {setup_info['gcs_key_path']}")
print(f"  Access: {'✓ Verified' if total >= 0 else '✗ Failed'}")

print(f"\nVideos:")
print(f"  Clear: {setup_info['video_count']['clear']}")
print(f"  Smash: {setup_info['video_count']['smash']}")
print(f"  Total: {setup_info['video_count']['total']}")

print(f"\n{'='*60}")

# Save setup info
with open('/content/setup_info.json', 'w') as f:
    json.dump(setup_info, f, indent=2, default=str)

print("\n✓ Setup information saved to /content/setup_info.json")

**✅ Checkpoint 8:** Setup complete!

---

## Step 9: Create Helper Script

Save environment variables for easy reuse in terminal.

In [ ]:
# Create environment setup script
env_script = f'''#!/bin/bash
# Environment setup for Colab
# Source this file before running scripts: source /content/setup_env.sh

export GOOGLE_APPLICATION_CREDENTIALS="{os.getenv('GOOGLE_APPLICATION_CREDENTIALS', '')}"
export GCS_BUCKET="{GCS_BUCKET}"
export PYTHONPATH="/content/iti123_v2:$PYTHONPATH"

# Activate virtual environment
source /content/iti123_v2/colab_venv/bin/activate

echo "✓ Environment configured"
echo "  GCS Bucket: $GCS_BUCKET"
echo "  Python: $(which python)"
echo "  Working dir: $(pwd)"
'''

with open('/content/setup_env.sh', 'w') as f:
    f.write(env_script)

!chmod +x /content/setup_env.sh

print("✓ Environment script created: /content/setup_env.sh")
print("\nTo use in terminal:")
print("  source /content/setup_env.sh")

**✅ Checkpoint 9:** Helper script created

---

## 🎉 Setup Complete!

### What We Did

✅ **Repository:** Cloned from GitHub  
✅ **Environment:** Python 3.10 virtual environment created  
✅ **Dependencies:** All packages installed  
✅ **GCS:** Authenticated and bucket verified  
✅ **Videos:** Verified access to video files  
✅ **Directories:** Local structure created  
✅ **Helper Script:** Environment variables saved  

### Next Steps

**Option 1: Use Complete Workflow Notebook**
- Open: `notebooks/complete_workflow_colab.ipynb`
- Runs all phases in sequence
- Duration: ~8-12 hours

**Option 2: Use Phase-Specific Notebooks**
1. **Phase 2:** `notebooks/phase2_validation_colab.ipynb` (3-5 hours)
2. **Phase 3:** `notebooks/phase3_model_training_colab.ipynb` (2-4 hours)
3. **Phase 4:** `notebooks/phase4_production_integration_colab.ipynb` (1-2 hours)

**Option 3: Use Command-Line Scripts**
```bash
# In Colab terminal
source /content/setup_env.sh
cd /content/iti123_v2

# Run complete workflow
bash scripts/colab_phase2_validation.sh
```

### Quick Commands

**Download videos:**
```python
!gsutil -m rsync -r gs://{GCS_BUCKET}/videos/clips/ data/videos/clips/
```

**Extract poses:**
```python
!colab_venv/bin/python scripts/extract_poses_parallel.py \
    --video-dir data/videos/clips \
    --output-dir data/processed/poses \
    --model-complexity 1 \
    --target-fps 20 \
    --num-workers 4
```

**Check status:**
```python
!bash scripts/check_extraction_status.sh
```

### Environment Info Saved

- Setup details: `/content/setup_info.json`
- Environment script: `/content/setup_env.sh`
- GCS bucket name: `/content/gcs_bucket_name.txt`
- GCS key path: `/content/gcs_key_path.txt`

### Troubleshooting

If you encounter issues:
1. **Re-authenticate:** Re-run Step 4
2. **Check bucket access:** Re-run Step 5
3. **Verify videos:** Re-run Step 6
4. **Review setup info:** `!cat /content/setup_info.json`

### Documentation

- **Workflow Overview:** `WORKFLOW_OVERVIEW.md`
- **Quick Start:** `COLAB_QUICKSTART.md`
- **Upload Guide:** `notebooks/COLAB_UPLOAD_GUIDE.md`
- **Scripts:** `scripts/README.md`

---

**🚀 Your environment is ready! Proceed to Phase 2 (Pose Extraction) →**